# Bidirectional LSTM

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences


num_words = 10000

(x_train, y_train), (x_test, y_test) = reuters.load_data(
    num_words=num_words
)

print("Training samples:", len(x_train))
print("Testing samples :", len(x_test))
print("Number of classes:", np.max(y_train) + 1)

Training samples: 8982
Testing samples : 2246
Number of classes: 46


In [2]:
max_length = 200

x_train = pad_sequences(
    x_train,
    maxlen=max_length
)

x_test = pad_sequences(
    x_test,
    maxlen=max_length
)

print("\nTraining shape:", x_train.shape)
print("Testing shape :", x_test.shape)


Training shape: (8982, 200)
Testing shape : (2246, 200)


In [3]:
# ============================================================
# BIDIRECTIONAL LSTM — REUTERS NEWS CLASSIFICATION
# ============================================================

with tf.device('/GPU:0'):

    model = keras.Sequential([

        layers.Input(shape=(200,)),
        # our batchsize may vary 32,64..
        # so our input per sequence is 200 after padding
        # thats constant (200) thats why (200,)


        layers.Embedding(
            input_dim=10000,
            output_dim=128
        ),
        # This Embedding layer will give each word its own
        # 128-dimensional embedding vector.
        #
        # input_dim=10000 means we are using the top 10000
        # words/tokens from the Reuters vocabulary.
        #
        # output_dim gives the size of vector representation
        # for each word.
        #
        # shape before embedding:
        # (batch_size,200)
        #
        # after Embedding:
        # (batch_size,200,128)


        layers.Bidirectional(
            layers.LSTM(
                64,
                return_sequences=True
            )
        ),
        # here 64 refers to the number of hidden units
        # in EACH LSTM direction.
        #
        # Bidirectional means we have TWO LSTMs:
        #
        # Forward LSTM:
        # x1 → x2 → x3 → ... → x200
        #
        # Backward LSTM:
        # x200 → x199 → x198 → ... → x1
        #
        # return_sequences=True because we are going to
        # stack another Bidirectional LSTM layer.
        #
        # So both directions will return their hidden state
        # for every timestep.
        #
        # Forward output:
        # (batch_size,200,64)
        #
        # Backward output:
        # (batch_size,200,64)
        #
        # Bidirectional combines both using concatenation:
        #
        # (batch_size,200,64)
        #             +
        # (batch_size,200,64)
        #             ↓
        # (batch_size,200,128)


        layers.Bidirectional(
            layers.LSTM(
                64
            )
        ),
        # here 64 again refers to the number of hidden units
        # in EACH direction.
        #
        # This second Bidirectional LSTM receives the complete
        # sequence from the first Bidirectional LSTM.
        #
        # Input shape:
        # (batch_size,200,128)
        #
        # Since return_sequences=False by default,
        # each LSTM will return only its final hidden state.
        #
        # Forward LSTM:
        # (batch_size,64)
        #
        # Backward LSTM:
        # (batch_size,64)
        #
        # Bidirectional concatenates both:
        #
        # (batch_size,64) + (batch_size,64)
        #              ↓
        # (batch_size,128)


        layers.Dense(
            64,
            activation='relu'
        ),
        # This is a fully connected layer.
        #
        # It receives the 128 values from the second
        # Bidirectional LSTM.
        #
        # 128 → 64
        #
        # ReLU activation is used to introduce non-linearity.
        #
        # shape:
        # (batch_size,128)
        #        ↓
        # (batch_size,64)


        layers.Dense(
            46,
            activation='softmax'
        )
        # Reuters has 46 different topic classes.
        #
        # So this output layer has 46 neurons.
        #
        # Softmax converts these 46 values into probabilities.
        #
        # Example:
        #
        # Class 0  → 0.02
        # Class 1  → 0.01
        # Class 2  → 0.73
        # Class 3  → 0.04
        # ...
        # Class 45 → 0.01
        #
        # The probabilities of all 46 classes add up to 1.
        #
        # The class with the highest probability becomes
        # our predicted Reuters topic.
        #
        # shape:
        # (batch_size,64)
        #        ↓
        # (batch_size,46)

    ])


    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

In [4]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=25,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/25
113/113 [==============================] - 16s 82ms/step - loss: 2.2111 - accuracy: 0.4463 - val_loss: 1.7566 - val_accuracy: 0.5448
Epoch 2/25
113/113 [==============================] - 8s 70ms/step - loss: 1.7102 - accuracy: 0.5624 - val_loss: 1.6374 - val_accuracy: 0.5854
Epoch 3/25
113/113 [==============================] - 8s 70ms/step - loss: 1.5706 - accuracy: 0.6057 - val_loss: 1.6262 - val_accuracy: 0.5982
Epoch 4/25
113/113 [==============================] - 8s 70ms/step - loss: 1.4153 - accuracy: 0.6454 - val_loss: 1.5306 - val_accuracy: 0.6221
Epoch 5/25
113/113 [==============================] - 8s 69ms/step - loss: 1.2517 - accuracy: 0.6697 - val_loss: 1.4964 - val_accuracy: 0.6294
Epoch 6/25
113/113 [==============================] - 8s 69ms/step - loss: 1.1348 - accuracy: 0.6980 - val_loss: 2.0525 - val_accuracy: 0.5815
Epoch 7/25
113/113 [==============================] - 8s 69ms/step - loss: 1.3988 - accuracy: 0.6558 - val_loss: 1.6062 - val_accuracy: 0.619

In [5]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    batch_size=64
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

36/36 [==============================] - 1s 33ms/step - loss: 1.5662 - accuracy: 0.6131
Test Loss: 1.5662333965301514
Test Accuracy: 0.613089919090271


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

texts = [
    "The stock market is experiencing significant gains today.",
    "Scientists have discovered a new species in the Amazon rainforest.",
    "The football match ended with a surprising score of 3-1."
]

max_features = 10000
maxlen = 200

# Get Reuters vocabulary
word_index = reuters.get_word_index()

# Create tokenizer
tokenizer = Tokenizer(
    num_words=max_features,
    oov_token="<OOV>"
)

# Use Reuters vocabulary
tokenizer.word_index = {
    word: index + 3
    for word, index in word_index.items()
}

tokenizer.word_index["<OOV>"] = 2

# Create reverse word index
tokenizer.index_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}

# Convert text into Reuters word IDs
sequences = tokenizer.texts_to_sequences(texts)

# Display encoded sequences
for text, sequence in zip(texts, sequences):
    print("Original:", text)
    print("Encoded :", sequence)
    print()

# Pad sequences
padded_sequences = pad_sequences(
    sequences,
    maxlen=maxlen
)

print("Padded shape:", padded_sequences.shape)

# Predict
predictions = model.predict(
    padded_sequences,
    verbose=0
)

# Reuters class names
class_names = [
    "cocoa", "grain", "veg-oil", "earn", "acq",
    "wheat", "copper", "housing", "money-supply", "coffee",
    "sugar", "trade", "reserves", "ship", "cotton",
    "carcass", "crude", "nat-gas", "cpi", "money-fx",
    "interest", "gnp", "meal-feed", "alum", "oilseed",
    "gold", "tin", "strategic-metal", "livestock", "retail",
    "ipi", "iron-steel", "rubber", "heat", "jobs",
    "lei", "bop", "zinc", "orange", "pet-chem",
    "dlr", "gas", "silver", "wpi", "hog", "lead"
]

# Display predictions
for text, prediction in zip(texts, predictions):
    predicted_class = np.argmax(prediction)
    confidence = prediction[predicted_class]

    print("Text:", text)
    print("Predicted Topic:", class_names[predicted_class])
    print("Predicted Class:", predicted_class)
    print("Confidence:", f"{confidence:.4f}")
    print()

Original: The stock market is experiencing significant gains today.
Encoded : [4, 80, 76, 23, 7445, 922, 629, 145]

Original: Scientists have discovered a new species in the Amazon rainforest.
Encoded : [6849, 57, 3757, 10, 73, 2, 7, 4, 2, 2]

Original: The football match ended with a surprising score of 3-1.
Encoded : [4, 2, 2726, 273, 31, 10, 7932, 2, 5, 12, 19]

Padded shape: (3, 200)
Text: The stock market is experiencing significant gains today.
Predicted Topic: earn
Predicted Class: 3
Confidence: 0.7366

Text: Scientists have discovered a new species in the Amazon rainforest.
Predicted Topic: earn
Predicted Class: 3
Confidence: 0.6282

Text: The football match ended with a surprising score of 3-1.
Predicted Topic: earn
Predicted Class: 3
Confidence: 0.7408

